In [ ]:
# Py2JSON quick demo
# This cell is intentionally a code cell that prints the notebook header when run.
print('py2json quick demo notebook')

# Outline:
# - Imports and quick intros
# - Ctor examples (to_ctor_dict / construct / deconstruct)
# - fakit examples (load by dotpath and execute)
# - obj2dict / func_info_dict examples
# - encoders demo (numpy/bytes handling)


In [1]:
# Imports
from py2json.ctor import Ctor
from py2json.fakit import fakit, dotpath_to_obj, obj_to_dotpath
from py2json.obj2dict import func_info_dict
from py2json.encoders import dumps

print('Imported py2json modules')

# Quick: show Ctor.to_ctor_dict for collections.namedtuple
from collections import namedtuple
ctor_jdict = Ctor.to_ctor_dict(namedtuple, args=('A', 'x y z'), kwargs={'defaults': ('has', 'defaults')})
print('Ctor jdict for namedtuple:', ctor_jdict)

# Construct it
A = Ctor.construct(ctor_jdict)
print('Constructed A type:', A)
print('A signature:', func_info_dict(A).get('signature'))
print('Instantiate A:', A('x','y','z'))


Imported py2json modules
Ctor jdict for namedtuple: {'CONSTRUCTOR': {'module': 'collections', 'name': 'namedtuple', 'attr': None}, 'ARGS': ('A', 'x y z'), 'KWARGS': {'defaults': ('has', 'defaults')}}
Constructed A type: <class 'py2json.ctor.A'>
A signature: {'parameters': [{'name': 'x'}, {'name': 'y'}, {'name': 'z'}]}
Instantiate A: A(x='x', y='y', z='z')


In [2]:
# More demos: functions, partials, and fakit
from functools import partial

# Deconstruct a simple function (use built-in pow)
from math import pow
pow_ctor = Ctor.deconstruct(pow, output_type=Ctor.CTOR_DICT)
print('pow ctor dict:', pow_ctor)
print('pow reconstructed equals pow?:', Ctor.construct(pow_ctor) is pow)

# Partial example
p = partial(pow, 2)
p_ctor = Ctor.deconstruct(p, output_type=Ctor.JSON_DICT)
print('partial serialized (json-friendly):', p_ctor)
# reconstruct
p_recon = Ctor.construct(Ctor.deserializer(p_ctor))
print('partial recon callable result:', p_recon(10))

# Class instance with to_jdict/from_jdict example
try:
    from py2json.tests.test_ctor import MockToJdict
    inst = MockToJdict('test_value')
    inst_ctor = Ctor.deconstruct(inst)
    print('MockToJdict ctor:', inst_ctor)
    print('reconstructed:', Ctor.construct(inst_ctor))
except Exception as e:
    print('MockToJdict not available in tests; creating a simple class with to_jdict/from_jdict')
    class Simple:
        def __init__(self, value):
            self.value = value
        def to_jdict(self):
            return {'value': self.value}
        @classmethod
        def from_jdict(cls, d):
            return cls(d['value'])

    s = Simple('hello')
    s_ctor = Ctor.deconstruct(s)
    print('Simple ctor:', s_ctor)
    print('reconstructed simple:', Ctor.construct(s_ctor))

# fakit examples
print('\nFakit examples:')
print('join path fakit:', fakit({'f': 'os.path.join', 'a': ['I','am','a','filepath']}))
A = fakit(['collections.namedtuple', ('A','x y z')])
print('namedtuple type from fakit:', A)
print('A(1,2,3):', A(1,2,3))


pow ctor dict: {'CONSTRUCTOR': <function dill_load_string at 0x107016200>, 'ARGS': ['\x80\x04\x95\x10\x00\x00\x00\x00\x00\x00\x00\x8c\x04math\x94\x8c\x03pow\x94\x93\x94.'], 'KWARGS': None}
pow reconstructed equals pow?: True
partial serialized (json-friendly): {'CONSTRUCTOR': {'module': 'functools', 'name': 'partial', 'attr': None}, 'ARGS': [{'CONSTRUCTOR': {'module': 'py2json.ctor', 'name': 'dill_load_string', 'attr': None}, 'ARGS': ['\x80\x04\x95\x10\x00\x00\x00\x00\x00\x00\x00\x8c\x04math\x94\x8c\x03pow\x94\x93\x94.'], 'KWARGS': None}, 2], 'KWARGS': {}}
partial recon callable result: 1024.0
MockToJdict ctor: {'CONSTRUCTOR': {'module': 'py2json.tests.test_ctor', 'name': 'MockToJdict', 'attr': 'from_jdict'}, 'ARGS': [{'value': 'test_value'}], 'KWARGS': None}
reconstructed: <MockToJdict value='test_value'>

Fakit examples:
join path fakit: I/am/a/filepath
namedtuple type from fakit: <class 'py2json.fakit.A'>
A(1,2,3): A(x=1, y=2, z=3)


In [3]:
# obj2dict / func_info_dict demo

def example(a, b=2, *, c: int = 3) -> int:
    """Example function docstring"""
    return a + b * c

info = func_info_dict(example)
print('func_info_dict(example):', info)

# encoders demo: numpy and bytes
import numpy as np
arr = np.array([1,2,3])
b = b'hello'
print('dumps arr:', dumps({'arr': arr}))
print('dumps bytes:', dumps({'b': b}))

# MARKDOWN-style summary (as a python print to keep it runnable)
print('\nSUMMARY:')
print(' - dotpath forms used in the package: module.path.attr (fakit uses dotted module paths).')
print(' - Ctor uses a CtorDict with keys CONSTRUCTOR, ARGS, KWARGS. CONSTRUCTOR may serialize to {module,name,attr} jdict.')
print(' - fakit expresses calls as (f, a, k) or {f:a:k} or special $fak syntax for nested evaluation.')
print('\nNext: If you like this notebook I can add a README.md section explaining the dotpath language, recommended formats such as:')
print("  * 'module.submodule:attr' (colon separates module from in-module attribute)")
print("  * 'module.submodule.attr' (dot-only; will try to import progressively and then getattr)")
print("  * fakit $fak form for nested calls and to express arguments")


func_info_dict(example): {'name': 'example', 'signature': {'parameters': [{'name': 'a'}, {'name': 'b'}, {'name': 'c'}]}}
dumps arr: {"arr":[1,2,3]}
dumps bytes: {"b":"aGVsbG8="}

SUMMARY:
 - dotpath forms used in the package: module.path.attr (fakit uses dotted module paths).
 - Ctor uses a CtorDict with keys CONSTRUCTOR, ARGS, KWARGS. CONSTRUCTOR may serialize to {module,name,attr} jdict.
 - fakit expresses calls as (f, a, k) or {f:a:k} or special $fak syntax for nested evaluation.

Next: If you like this notebook I can add a README.md section explaining the dotpath language, recommended formats such as:
  * 'module.submodule:attr' (colon separates module from in-module attribute)
  * 'module.submodule.attr' (dot-only; will try to import progressively and then getattr)
  * fakit $fak form for nested calls and to express arguments
